<a href="https://colab.research.google.com/github/vivaan3141/ClarifyEdu/blob/main/Problem3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load dataset
try:
    df = pd.read_csv('uc_freshman_admission_by_discipline.csv')
except FileNotFoundError:
    df = pd.read_csv('Data/uc_freshman_admission_by_discipline.csv')

# Clean numeric columns if they are formatted as strings
for col in ['applicants', 'admits', 'enrollees']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Inspect column names to be dynamic
year_col = [c for c in df.columns if 'year' in c.lower() or 'term' in c.lower()][0]
campus_col = [c for c in df.columns if 'campus' in c.lower()][0]
disc_col = [c for c in df.columns if 'disc' in c.lower() or 'major' in c.lower()][0]

# Filter for Fall 2025 and exclude systemwide totals
df_2025 = df[(df[year_col] == 2025) & (~df[campus_col].str.contains('systemwide|universitywide', case=False, na=False))]

# 1. Calculate overall admit rate per campus
overall = df_2025.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())

# 2. Calculate Computer Science admit rate per campus
cs_df = df_2025[df_2025[disc_col].str.contains('Computer Science', case=False, na=False)]
cs_rates = cs_df.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())

# 3. Calculate the difference (Overall Rate - CS Rate)
rate_penalty = (overall - cs_rates).dropna().sort_values(ascending=False)

print("--- ADMIT RATE DIFFERENCE (Overall - CS) BY CAMPUS ---")
for campus, diff in rate_penalty.items():
    print(f"{campus}: Overall {overall[campus]:.2%} vs CS {cs_rates[campus]:.2%} (Cost: -{diff*100:.2f}%)")

print("\nQuestion 3 Answer (Campus):", rate_penalty.index[0])

--- ADMIT RATE DIFFERENCE (Overall - CS) BY CAMPUS ---
Davis: Overall 44.30% vs CS 19.33% (Cost: -24.97%)
San Diego: Overall 28.13% vs CS 19.87% (Cost: -8.26%)
Riverside: Overall 86.52% vs CS 81.12% (Cost: -5.41%)
Berkeley: Overall 11.32% vs CS 6.45% (Cost: -4.87%)
Santa Barbara: Overall 38.21% vs CS 33.91% (Cost: -4.29%)
Los Angeles: Overall 9.59% vs CS 7.32% (Cost: -2.27%)
Irvine: Overall 29.35% vs CS 27.58% (Cost: -1.77%)
Santa Cruz: Overall 72.51% vs CS 79.39% (Cost: --6.87%)

Question 3 Answer (Campus): Davis


/tmp/ipykernel_2047/641274511.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall = df_2025.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())
/tmp/ipykernel_2047/641274511.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cs_rates = cs_df.groupby(campus_col).apply(lambda g: g['admits'].sum() / g['applicants'].sum())
